# Lab 2.1: Agents with the Azure AI Foundry Agent Service (20 min)

## Objectives
- Create a **standard agent** in Azure AI Foundry (visible in the portal!)
- Understand the cycle: **Agent → Conversation → Response**
- Use **function calling** with agents
- Use the **Code Interpreter**
- Manage the agent lifecycle (create version, converse, delete)

## Concepts

### Agent vs Model Call
In Lab 2 we used the **Responses API** — direct calls to the model. Here we'll create **standard agents** with the **Agent Service**, visible in the Foundry portal:

| Responses API (Lab 2) | Agent Service (Lab 2.1) |
|---|---|
| Stateless call | Persistent agent in Foundry |
| No memory between calls | `previous_response_id` or Conversations maintain context |
| You manage the tool loop | The agent_reference dispatches configured tools |
| Ideal for simple use | Ideal for complex assistants |

### Lifecycle (Standard Agents)
```
1. Create Agent (create_version with PromptAgentDefinition) → appears in Foundry portal
2. Invoke via Responses API with agent_reference
3. Process tool calls (if any) and submit results
4. Continue conversation with previous_response_id or Conversations
5. Delete agent when no longer needed
```

### SDK — `azure-ai-projects` >= 2.0.0
```python
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Create standard agent
agent = project.agents.create_version(
    agent_name="my-agent",
    definition=PromptAgentDefinition(model="gpt-4o", instructions="..."),
)

# Invoke via OpenAI Responses API
openai = project.get_openai_client()
response = openai.responses.create(
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="Hello!",
)
```

In [ ]:
# Setup
from dotenv import load_dotenv
import os
import json

load_dotenv("../.env")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, CodeInterpreterTool
from azure.identity import DefaultAzureCredential

project_endpoint = os.getenv("AZURE_AI_AGENT_ENDPOINT")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

# Project client — to create/manage agents
project = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

# OpenAI client — to invoke agents via Responses API
openai_client = project.get_openai_client()

print(f"✅ Project Client connected: {project_endpoint}")

✅ Project Client conectado: https://foundry-mod2.services.ai.azure.com/api/projects/proj-tutor


## 🖥️ Exercise 2.1.1: Simple Agent

Let's create a standard agent with `PromptAgentDefinition` and invoke it via the **Responses API** with `agent_reference`.

In [ ]:
# Exercise 2.1.1: Create a standard agent (visible in the Foundry portal!)
agent = project.agents.create_version(
    agent_name="assistente-azure",
    definition=PromptAgentDefinition(
        model=model,
        instructions="You are an Azure specialist. Respond concisely with practical examples.",
    ),
)
print(f"✅ Agent created: {agent.name} (version {agent.version})")
print(f"   Model: {agent.definition.model}")
print(f"\n💡 Go to the Foundry portal and verify the agent appears there!")

✅ Agente criado: assistente-azure (versão 1)
   Modelo: gpt-4o

💡 Vai ao portal do Foundry e verifica que o agente aparece lá!


In [ ]:
# Invoke the agent via Responses API with agent_reference
response = openai_client.responses.create(
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="Explain the difference between Azure Functions and Container Apps in 3 lines.",
)
print(f"👤 Explain the difference between Azure Functions and Container Apps in 3 lines.")
print(f"🤖 {response.output_text}")
print(f"\n   Response ID: {response.id}")

👤 Explica a diferença entre Azure Functions e Container Apps em 3 linhas.
🤖 Azure Functions é um serviço de computação serverless para executar pequenas funções de forma escalável, ideal para tarefas event-driven (e.g., processar uma mensagem de fila). Azure Container Apps permite implementar aplicações completas em contentores, oferecendo maior controlo sobre configurações e ambientes personalizados. Functions é melhor para lógica pequena, enquanto Container Apps suporta aplicações mais complexas.

   Response ID: resp_005c1fc0dbc9c7d50069dd34947610819493ac19464961a5da


In [ ]:
# Continue the conversation using previous_response_id (the agent maintains context!)
follow_up = openai_client.responses.create(
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    previous_response_id=response.id,
    input="Which of the two would you recommend for a simple webhook?",
)
print(f"👤 Which of the two would you recommend for a simple webhook?")
print(f"🤖 {follow_up.output_text}")

👤 E qual dos dois recomendas para um webhook simples?
🤖 Para um webhook simples, recomendo **Azure Functions**, pois é otimizado para tarefas leves e event-driven, como responder rapidamente a chamadas HTTP. Além disso, reduz custos e complexidade, já que apenas pagas pelo tempo de execução.


## 🖥️ Exercise 2.1.2: Agent with Function Calling

The real power of agents is using **tools**. Let's create an agent with `FunctionTool` that can look up Azure pricing and region information. The cycle is: invoke → process tool calls → submit results → get final response.

In [ ]:
# Exercise 2.1.2: Define functions and create agent with tools
from openai.types.responses.response_input_param import FunctionCallOutput

# Functions the agent can call
def get_service_price(service: str) -> str:
    """Returns the estimated price of an Azure service."""
    prices = {
        "app service basic": "€50/month",
        "container apps": "€0.000012/vCPU-s",
        "functions consumption": "€0.000016/GB-s",
        "aks": "Free (control plane) + node costs",
        "cosmos db": "Starting at €25/month (serverless)",
    }
    return json.dumps({"service": service, "price": prices.get(service.lower(), "Price not available. Check azure.com/pricing")})

def get_recommended_region(use_case: str) -> str:
    """Recommends the best Azure region for a use case."""
    regions = {
        "europe": "West Europe (Netherlands) or North Europe (Ireland)",
        "portugal": "West Europe (closest to Portugal)",
        "global": "Use Traffic Manager with multiple regions",
        "ai": "East US or Sweden Central (best model availability)",
    }
    return json.dumps({"use_case": use_case, "recommendation": regions.get(use_case.lower(), regions["europe"])})

# Function dispatch map
func_map = {
    "get_service_price": get_service_price,
    "get_recommended_region": get_recommended_region,
}

# Define tools with explicit schemas (FunctionTool from azure-ai-projects)
tools = [
    FunctionTool(
        name="get_service_price",
        description="Returns the estimated price of an Azure service.",
        parameters={
            "type": "object",
            "properties": {"service": {"type": "string", "description": "Azure service name"}},
            "required": ["service"],
            "additionalProperties": False,
        },
        strict=True,
    ),
    FunctionTool(
        name="get_recommended_region",
        description="Recommends the best Azure region for a use case.",
        parameters={
            "type": "object",
            "properties": {"use_case": {"type": "string", "description": "The use case (e.g.: portugal, europe, ai)"}},
            "required": ["use_case"],
            "additionalProperties": False,
        },
        strict=True,
    ),
]

# Create agent with tools
agent_tools = project.agents.create_version(
    agent_name="consultor-azure",
    definition=PromptAgentDefinition(
        model=model,
        instructions="You are an Azure consultant. Use the available tools to provide accurate information about prices and regions.",
        tools=tools,
    ),
)
print(f"✅ Agent with tools created: {agent_tools.name} (version {agent_tools.version})")

✅ Agente com tools criado: consultor-azure (versão 1)


In [ ]:
# Test the agent with function calling
response_tools = openai_client.responses.create(
    extra_body={"agent_reference": {"name": agent_tools.name, "type": "agent_reference"}},
    input="I want to create an app with Container Apps. How much does it cost and what's the best region for Portugal?",
)

# Process tool calls and submit results
tool_outputs = []
for item in response_tools.output:
    if item.type == "function_call":
        print(f"[Tool] {item.name}({item.arguments})")
        result = func_map[item.name](**json.loads(item.arguments))
        tool_outputs.append(
            FunctionCallOutput(type="function_call_output", call_id=item.call_id, output=result)
        )

# Submit results and get final response
if tool_outputs:
    response_tools = openai_client.responses.create(
        input=tool_outputs,
        previous_response_id=response_tools.id,
        extra_body={"agent_reference": {"name": agent_tools.name, "type": "agent_reference"}},
    )

print(f"\n👤 I want to create an app with Container Apps. How much does it cost and what's the best region for Portugal?")
print(f"🤖 {response_tools.output_text}")

[Tool] obter_preco_servico({"servico":"Container Apps"})
[Tool] obter_regiao_recomendada({"caso_uso":"portugal"})

👤 Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?
🤖 Os preços para o serviço **Container Apps** no Azure começam em **€0.000012 por segundo de CPU (vCPU-s)**. O custo final dependerá do número de instâncias, tempo de execução e outros componentes que utilizar na sua aplicação.

A região recomendada para Portugal é **West Europe**, pois é a mais próxima e oferece baixa latência.


## 🖥️ Exercise 2.1.3: Code Interpreter

The **Code Interpreter** allows the agent to write and execute Python code autonomously. Just add it as a tool in the agent definition.

In [ ]:
# Exercise 2.1.3: Agent with Code Interpreter
agent_code = project.agents.create_version(
    agent_name="analista-dados",
    definition=PromptAgentDefinition(
        model=model,
        instructions="You are a data analyst. Use the code interpreter to perform calculations and create visualizations.",
        tools=[CodeInterpreterTool()],
    ),
)
print(f"✅ Agent with code interpreter: {agent_code.name} (version {agent_code.version})")

response_code = openai_client.responses.create(
    extra_body={"agent_reference": {"name": agent_code.name, "type": "agent_reference"}},
    input="Calculate the Fibonacci sequence up to the 10th number and show the result in a formatted table.",
)
print(f"\n🤖 {response_code.output_text}")

✅ Agente com code interpreter: analista-dados (versão 1)

🤖 Aqui está a tabela com os números da sequência de Fibonacci até ao 10º número:

| Posição (n) | Número de Fibonacci |
|-------------|---------------------|
| 1           | 0                   |
| 2           | 1                   |
| 3           | 1                   |
| 4           | 2                   |
| 5           | 3                   |
| 6           | 5                   |
| 7           | 8                   |
| 8           | 13                  |
| 9           | 21                  |
| 10          | 34                  |


## 🧹 Cleanup

Agents are **persistent** — it's important to delete them when they're no longer needed.

In [ ]:
# Cleanup - delete all created agents
for name in ["assistente-azure", "consultor-azure", "analista-dados"]:
    try:
        project.agents.delete(agent_name=name)
        print(f"🗑️ Agent '{name}' deleted")
    except Exception as e:
        print(f"⚠️ Error deleting '{name}': {e}")

print("\n✅ Cleanup complete! Check the Foundry portal to verify the agents are gone.")

🗑️ Agente 'assistente-azure' eliminado
🗑️ Agente 'consultor-azure' eliminado
🗑️ Agente 'analista-dados' eliminado

✅ Limpeza concluída! Verifica no portal do Foundry que os agentes desapareceram.


## ✅ Summary

In this lab you learned:
- How to create **standard agents** with `PromptAgentDefinition` via `azure-ai-projects` >= 2.0.0
- Invoking agents via the **Responses API** with `agent_reference`
- Maintaining **context** between turns using `previous_response_id`
- Using **function calling** with the cycle: invoke → process tool calls → submit results
- Using the **Code Interpreter** for autonomous computation
- Managing the lifecycle (create version and delete agents)

### Lab 2 vs Lab 2.1
| Lab 2 (Responses API) | Lab 2.1 (Agent Service) |
|---|---|
| Direct calls to the model | Standard agents in the Foundry portal |
| You manage the context | `previous_response_id` or Conversations |
| Tools defined inline | Tools associated with the agent (persistent) |
| Simpler and lighter | More powerful for complex assistants |

**Next:** [Lab 3 - LLM Workflows](../lab03/lab03-model-workflows.ipynb)